<p style="font-size: 40px; text-align: center;">Efficient Use of Cluster Misha - Part 1</p>

# Cluster in a Nut Shell

## Hardware Components

A **cluster** is a group of interconnected computers that work together as a single system to perform computational tasks. Each computer is referred to as a **node**. A **login node** serves as a shared access point, allowing users to connect to the cluster to edit code, submit jobs, and manage files. The login node is not meant for running heavy computations; instead, it’s used for preparing and launching jobs to the compute nodes. A **compute node** is a dedicated node where user jobs actually run. Compute nodes provide the CPUs, GPUs, and memory requested in a job submission. They are managed by the batch scheduler and are optimized for computational tasks, not for direct interactive use. A  cluster typically consists of several login nodes and many compute nodes. The number of compute nodes can range from dozens to tens of thousands, depending on the system’s scale. The cluster also includes shared storage for data. All components — including login nodes, compute nodes, and storage — are interconnected through a network to form a unified system.  

A cluster is often referred to as an **High-Performance Computing** (HPC) cluster when its nodes are equipped with high-end CPUs and GPUs, feature large memory capacities, share a fast, high-capacity storage system, and are interconnected through a high-speed network. Such clusters are designed to efficiently handle large-scale, multi-node computations and data-intensive workloads.

<center><img src="figures/cluster_diagram.png" width="600"></center>

## Software Components

* Linux - the Operating System
* Batch system - job and computing resource management 
* Software modules - application management
* Open OnDemand - HPC web portal

## In Real Life

### Cluster nodes
<center> <img src="figures/cluster_nodes.png" width="600"> </center>

### Cluster
<center> <img src="figures/cluster_misha.png" width="400"> </center>


# Cluster Misha

## Inspiration Behind Its Name

<center> <img src="figures/misha_mahowald.png" width="200"> </center>

**Misha Mahowald** was a pioneering neuroscientist and engineer whose visionary work helped bridge biology and computation. She is best known for developing the first analog silicon retina and for her groundbreaking contributions to neuromorphic engineering — the design of electronic systems inspired by the brain. The Misha cluster is named in her honor, celebrating her legacy of innovation at the intersection of neuroscience and technology, and inspiring the next generation of computational discovery. 

## Storage

| Name | Location | Quota | File Limit | Type | Expandable | Backup | Purged |
|------|----------|-------|------------|------|------------|--------|--------|
| home | /gpfs/radev/home/netid | 125 GiB | 500,000 | user | no | yes | no |
|project|/gpfs/radev/project/group/netid| 4 TiB | 5,000,000 | group | no | no | no |
|scratch|/gpfs/radev/scratch/group/netid| 10 TiB | 15,000,000 | group | maybe| no| 60 days |
|pi_group| /gpfs/radev/pi/group/netid<br>/gpfs/marilyn/pi/group/netid| varied | varied | group | yes | no | no |

A PI can apply for additional storage for their group by filling out this [form](https://docs.google.com/forms/d/e/1FAIpQLSeer4iwemFWn7ODZnIfP-qiX7j54veFdjRfRPVFlYAS4tQsdg/viewform?usp=dialog). 

In [ ]:
getquota

## Nodes

Node types and quantities:

| Node Type | # of Nodes | CPU Cores/Node| CPU Type | Memory |
|-------|------------|-------|-----------|----------|
| Login| 2 | 32 | Intel Xeon Gold 6326 (Ice Lake) | 512 GB|
| CPU| 26 | 64| Intel Xeon Gold 6458Q (Sapphire Rapids) | 512 GB |
| Bigmem| 2 | 64 | Intel Xeon Gold 6458Q (Sapphire Rapids) | 2 TB |
| GPU | 33   | 32 or 48 | varies (see the table blow) | 1 TB |

GPU types and quantities:

| GPU type | # of Nodes | # of Cards | VRAM/card | CPU Cores/Node | CPU Type |
|------|----------|-------|------------|-------|-------|
| H200 | 4 | 16 | 141 GB |  48 | Intel Xeon Gold 6542Y (Emerald Rapids) |
| H100 |14 | 56 | 80 GB  |  48 | Intel Xeon Gold 6542Y (Emerald Rapids) |
| A100 | 6 | 16 | 80 GB  |  32 | Intel Xeon Gold 6326 (Ice Lake) |
| L40S | 3 | 12 | 48 GB  |  32 | Intel(R) Xeon(R) Gold 6442Y (Sapphire Rapids |
| A40  | 4 | 16 | 48 GB  |  48 | Intel Xeon Gold 6326 (Ice Lake) |


## SLURM - Job Scheduler and Resource Manager
SLURM (Simple Linux Utility for Resource Management) is an open-source workload manager widely used on HPC clusters to schedule and allocate computational resources such as CPUs, GPUs, and memory. It manages job queues, assigns tasks to available nodes, monitors job progress, and enforces user and group policies for fair resource sharing. SLURM allows researchers to run single or parallel jobs efficiently across many nodes, supporting MPI, OpenMP, and hybrid workflows. 
### Partitions
In SLURM, a partition is a logical grouping of compute nodes that defines where and how jobs can run. Each partition can represent a different type of resource or usage policy—for example, separate partitions for GPU nodes, high-memory nodes, or general-purpose nodes. Partitions help system administrators control access, set job limits, and manage priorities across diverse hardware. Users specify a partition when submitting a job to ensure it runs on the appropriate resources (e.g., --partition=gpu). 

Misha has a simple partition structure with six public partitions that are accessible to all users. Contributing group members can access their resources through QOS, not through a private partition, which we will examine later. 

The table below shows the configuration of each partition. The per-user limit is an aggregated limit for each user, and the per-group limit is an aggregated limit for each group. The limits for the **gpu** partition are dynamically changed based on the loads on the cluster. 

| Name      |  Nodes   |    Time Limit |  Per-user Limit   | Per-group Limit |
|-----------|----------|---------------|--------------------|--------------------|
| devel     |   2      |     6 hrs     |  cpu=10,gres/gpu=4,mem=70G|  |
|  day      |   18     |     24 hrs    | cpu=512,mem=20T | cpu=1024,mem=37.50T  |
| bigmem    |   2      |     24 hrs    | cpu=64,mem=2T | |
| week      |   6      |     7 days    |  cpu=128,mem=1280G | cpu=192,mem=1920G  |
| gpu_devel |   2      |     6 hrs     |  cpu=4,gres/gpu=1,mem=32G |  |
| gpu       |   31     |     48 hrs    |  cpu=192,gres/gpu=18 | cpu=384,gres/gpu=36 |

In [ ]:
# Diaplay quick information about partitions and node states
sinfo

### Jobs
In SLURM, a job is a user-submitted request to run one or more computational tasks on the cluster. Each job specifies details such as the number of CPUs or GPUs, memory requirements, runtime limits, and the partition to run on. Jobs can be simple single-node tasks or large-scale parallel workloads spanning many nodes. Users typically submit jobs using the `sbatch` command for batch execution or `salloc` for interactive runs. Once submitted, SLURM places the job in a queue, schedules it based on available resources and priority, and monitors its progress until completion. This job-based model ensures efficient use of shared resources, reproducibility of computations, and fair access among multiple users.

#### Common job options
A job specifies its requirements using SLURM job options. Commonly used job options are listed below:

| Option               | Example                       | Description                                       |
| -------------------- | ----------------------------- | ------------------------------------------------- |
| `--job-name`         | `#SBATCH --job-name=myjob`    | Name of the job (shown in `squeue`).              |
| `--output`           | `#SBATCH --output=job_%j.out` | File for standard output (`%j` = job ID).         |
| `--error`            | `#SBATCH --error=job_%j.err`  | File for standard error.                          |
| `--time`             | `#SBATCH --time=01:00:00`     | Walltime limit (HH:MM:SS).                        |
| `--partition` (`-p`) | `#SBATCH -p gpu`              | Partition/queue to submit to.                     |
| `--nodes` (`-N`)     | `#SBATCH -N 2`                | Number of nodes required.                         |
| `--ntasks` (`-n`)    | `#SBATCH -n 8`                | Number of tasks (MPI ranks).                      |
| `--cpus-per-task`    | `#SBATCH --cpus-per-task=4`   | Threads per task (for OpenMP/multithreaded jobs). |
| `--mem`              | `#SBATCH --mem=64G`           | Memory per node.                                  |
| `--mem-per-cpu`      | `#SBATCH --mem-per-cpu=4G`    | Memory per CPU.                                   |
| `--gres`                              | `#SBATCH --gres=gpu:1`      | Request GPU(s). (Classic syntax.)          |
| `--gpus`                              | `#SBATCH --gpus=1`          | Newer, simpler syntax for requesting GPUs. |
| `--constraint` or `--constraint=h100` | `#SBATCH --constraint=h100` | Select specific GPU model or node type.    |
| `--qos`            | `#SBATCH --qos=normal`           | Quality of Service (priority level).                  |
| `--reservation`    | `#SBATCH --reservation=workshop` | Run inside a reservation.                             |
| `--requeue`        | `#SBATCH --requeue`              | Allow job to be requeued after preemption or failure. | 
| `--mail-type`  | `#SBATCH --mail-type=BEGIN,END,FAIL`       | Send email notifications.                                 |
| `--mail-user`  | `#SBATCH --mail-user=user@yale.edu`  | Address for notifications.                                |
| `--array`             | `#SBATCH --array=1-10` | Submit a job array (10 jobs with one script). |

#### Interactive job
An interactive job is launched with `salloc`. It gives you a live shell on a compute node — it’s like logging in to a node that SLURM allocates for you. You can then run commands, test code, or launch programs interactively, with full access to GPUs/CPUs.

```{bash}
salloc --mem=10g -N 1 -n 1 --time=1:00:00 --gpus=1 -p gpu_devel
```

#### Batch jobs
A batch job runs automatically using a job script submitted with `sbatch`. It’s the most common type — you prepare a script with your resource requests and commands, submit it, and SLURM runs it when resources become available.

A simple batch job example:

```{bash}
#!/bin/bash
#SBATCH -p gpu
#SBATCH --gres=gpu:1
#SBATCH --time=04:00:00
#SBATCH --mem=10g
#SBATCH --job-name=train_model
#SBATCH --output=train_%j.out

module load miniconda
conda activate myenv
python train_model.py
```
Submit the job script with 

```{bash}
sbatch train_model.job
```

#### Array jobs
An array job is a convenient way to submit many similar batch jobs with one job script. Each sub-job (array element) runs the same script as specified in the job script, but with a unique index value (`$SLURM_ARRAY_TASK_ID`).

More information about array jobs can be found in the [YCRC user guide](https://docs.ycrc.yale.edu/clusters-at-yale/job-scheduling/dsq/).

#### Job monitoring commands

| Command                     | Description                         |
| --------------------------- | ----------------------------------- |
| `squeue`                    | Show running/queued jobs.           |
| `scontrol show job <jobid>` | Show detailed job info.             |
| `sacct -j <jobid>`          | Show completed job accounting data. |
| `sinfo`                     | Display node/partition states.      |

In [ ]:
sinfo

In [ ]:
# View information about jobs located in the queue. 
squeue

In [ ]:
# show my jobs
squeue --me

## Modules

In HPC, software modules provide a convenient way to manage and access different software packages, libraries, and compilers without conflicts. The module system allows users to dynamically modify their environment—such as PATH, LD_LIBRARY_PATH, and other variables—by loading or unloading specific modules with simple commands like `module load CUDA` or `module unload CUDA`. This approach enables multiple versions of the same software to coexist, ensuring compatibility with different workflows and projects. By utilizing software modules, HPC systems maintain a clean, flexible, and reproducible environment that allows users to easily configure their setups for various applications and research needs.

In [ ]:
# Display all available modules installed on the cluster
module avail

In [ ]:
# List modules currently loaded
module list

In [ ]:
# Search for modules
module spider CUDA

In [ ]:
module load CUDA
module list

In [ ]:
module unload CUDA
module list

In [ ]:
module load CUDA GCC Python
module list

In [ ]:
module purge
module list

## Conda 
**Conda** is an open-source package and environment manager that simplifies the installation, updating, and management of software and dependencies across platforms. It allows users to create isolated environments, each with its own versions of Python and libraries, ensuring compatibility and reproducibility for different projects. 

By default, all downloaded Conda packages are stored in `$HOME/.conda/pkgs` and user-created Conda environments are installed in `$HOME/.conda/envs`. These two paths can be changed by setting two environment variables in `$HOME/.bashrc`:

```{bash}
export CONDA_ENV_PATH=/path/to/conda_envs:$HOME/.conda/envs
export CONDS_PKG_DIRS=/path/to/conda_pkgs:$HOME/.conda/pkgs
```
Storing Conda files in the default paths is simple to maintain. When the primary group of your account is changed, your Conda environments will not be affected because your home directory remains unchanged. If you install your Conda environments in the project folder or a PI fileset folder, a change of group often breaks your Conda environments. However, there is one caveat: because Conda files can quickly fill up your home directory, you may easily run out of disk space or reach the maximum file count.  We will see how to fix this later.

In [ ]:
sbatch --mem=30g 
module load miniconda
conda create -n 

In [ ]:
conda activate 

In [ ]:
conda env list

In [ ]:
conda list

In [ ]:
conda env rm

In [ ]:
conda env export -f 

In [ ]:
conda env install -f 

<div class="alert alert-block alert-warning">
    <b>Warning:</b> Do not activate a Conda environment in your `$HOME/.bashrc`. 
</div>
<div class="alert alert-block alert-warning">
    <b>Warning:</b> Do not run `conda init` in a bash command-line. 
</div>

# Misha OnDemand Portal

**Open OnDemand** is an open-source web portal that provides an easy, browser-based interface for accessing high-performance computing resources. It allows users to submit and monitor jobs, manage files, access cluster terminals, and run interactive applications such as Jupyter Notebooks, RStudio, and MATLAB—all without requiring Linux command-line expertise. By simplifying HPC access, Open OnDemand makes powerful computing clusters more user-friendly and widely accessible to researchers and students.

Misha OnDemand portal is available at [https://ood-misha.ycrc.yale.edu](https://ood-misha.ycrc.yale.edu).


* **Dashboard:** 
* **File manager:**
* **Job manager:**
* **Shell access:**
* **Interactive apps:**
*
* introduce the interface first - how they corresponds to the different options in SLURM

   * Jupyter
   * VSCode Proxy
You can run your local VSCode on a Misha compute node via a login node, which acts as an intermediate proxy to securely forward the SSH traffic. The process is simplified by using the Misha VSCode Proxy OOD app.
VSCode Proxy simplifies the process of running your local VSCode instance on a remote compute node in Misha.

Benefit: 1. Same environment, but utilizing cluster resources. 2. AI coding assistant
   * Code Server
   * RStudio Server
   * MATLAB
   * Remote Desktop
        * Visualization: ycrc_vglrun

Utility apps